# Doc2Vec - K-Means Topic Modelng Tutorial

In [6]:
# Required to display widgets in Jupyter
import os
import sys
import pandas as pd

#This will navigate up two levels in the directory structure to the project root directory. 
sys.path.append(os.path.dirname(os.getcwd()))

#Import to minimize warning. 
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
from topicminer import Doc2Vec_KMeans_Cat

In [21]:
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from topicminer import read_json_to_dataframe
from topicminer import PROCESSED_TEXT_COL
from gensim.models import Word2Vec
from sklearn.cluster import KMeans

In [25]:
class Word2VecKMeansCategorizer:
    def __init__(self, vector_size=100, window=5, min_count=5, workers=8, epochs=10, n_clusters=5):
        if workers > 8:
            print("Workers can be no higher than 8. Resetting to 8.")
            workers = 8
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.epochs = epochs
        self.workers = workers
        self.n_clusters = n_clusters
        self.model = None
        self.kmeans = None

    def train_word2vec_model(self, sentences):
        from gensim.models import Word2Vec
        self.model = Word2Vec(sentences, vector_size=self.vector_size, window=self.window,
                              min_count=self.min_count, workers=self.workers, epochs=self.epochs)

    def document_vector(self, doc):
        """Averaging word vectors in a document."""
        words = [word for word in doc if word in self.model.wv.index_to_key]
        if words:
            return np.mean(self.model.wv[words], axis=0)
        else:
            return np.zeros(self.vector_size)

    def categorize_documents(self):

        # Obtained tokenized docs
        tokenized_docs = read_json_to_dataframe(columns=[PROCESSED_TEXT_COL])[PROCESSED_TEXT_COL].to_list()
        
        # Train Word2Vec
        self.train_word2vec_model(tokenized_docs)
        
        # Create document vectors
        doc_vectors = np.array([self.document_vector(doc) for doc in tokenized_docs])
        
        # Cluster document vectors
        from sklearn.cluster import KMeans
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=0).fit(doc_vectors)
        
        # Extract top terms
        key_terms = self.extract_key_terms(tokenized_docs, doc_vectors)
        
        # Map cluster labels to key terms
        label_key_terms_dict = {i: key_terms[i] for i in range(self.n_clusters)}
        
        return label_key_terms_dict

    def extract_key_terms(self, docs, vectors):
        from sklearn.metrics.pairwise import cosine_similarity
        centroids = self.kmeans.cluster_centers_
        terms = {}
        for i in range(self.n_clusters):
            centroid = centroids[i]
            cos_similarities = cosine_similarity([centroid], vectors)
            top_docs_indices = np.argsort(cos_similarities[0])[-3:]
            cluster_terms = []
            for idx in top_docs_indices:
                cluster_terms.extend(docs[idx].split())
            unique_terms = list(set(cluster_terms))
            terms[i] = unique_terms[:5]  # Assume top 5 terms are key terms
        return terms


In [31]:
# Example usage
categorizer = Word2VecKMeansCategorizer(n_clusters=10)
labels, key_terms = categorizer.categorize_documents()

3165

In [20]:
tokenized_docs = read_json_to_dataframe(columns=[PROCESSED_TEXT_COL])[PROCESSED_TEXT_COL].to_list()
tokenized_docs

['internal announcement project prioritization alignment dear team as continue drive innovation growth cardiff electric wanted take moment discus project prioritization alignment with current pipeline initiative essential focus critical project align strategic objective to ensure working efficiently effectively please find key takeaway action item review project chart provide input eod friday identify top three priority project cob monday att project alignment meeting tuesday pm discus refine priority understand may require extra effort confident collective expertise yield positive result please let know question concern look forward reviewing input discussing next step tuesday best regard ed burris project manager cardiff electric phone',
 'social event invitation dear joe writing ext invitation att social event hosted cardiff electric legal department as part effort organized networking evening would delighted could join u the event take place friday march pm hyatt regency downtown t